In [5]:
#!/usr/bin/env python3

import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# USER SETTINGS
# ============================================================
NETID = os.environ.get("NETID", "k16v981")

DAILY_PEAK_GLOB = (
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/data/DailyPeakState/DailyPeakState-*.nc"
)

PHASE_CSV = (
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv"
)

OUT_DIR = Path(
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/figures/phase_maps"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_VAR = "t2m_at_wbt_daily_peak"
Q_VAR = "q_at_wbt_daily_peak"

LAT_MIN, LAT_MAX = 10, 34
LON_MIN, LON_MAX = 34, 60

PCTL = 0.95
MIN_COUNT = 10

ENSO_LAG = 2
IOD_LAG = 1

USE_MONTHS = [6, 7, 8, 9]

ENSO_POS_THRESH = 0.5
ENSO_NEG_THRESH = -0.5
IOD_POS_THRESH = 0.5
IOD_NEG_THRESH = -0.5

# ============================================================
# STYLES
# ============================================================
ERL_RC = {
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 8,
    "path.simplify": False,
    "savefig.transparent": False,
}

PANEL_LABELS = [
    "(a)", "(b)", "(c)", "(d)",
    "(e)", "(f)", "(g)", "(h)",
    "(i)", "(j)", "(k)", "(l)",
]

# ============================================================
# HELPERS
# ============================================================
def classify_enso_from_roni(val):
    if pd.isna(val):
        return np.nan
    if val >= ENSO_POS_THRESH:
        return "El Nino"
    if val <= ENSO_NEG_THRESH:
        return "La Nina"
    return "Neutral"


def classify_iod_from_dmi(val):
    if pd.isna(val):
        return np.nan
    if val >= IOD_POS_THRESH:
        return "pIOD"
    if val <= IOD_NEG_THRESH:
        return "nIOD"
    return "Neutral"


def standardize_time_dim(ds: xr.Dataset) -> xr.Dataset:
    if "day" in ds.dims:
        ds = ds.rename({"day": "time"})
    if "day" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"day": "time"})

    ds = ds.sortby("latitude")
    ds = ds.sortby("longitude")

    ds = ds.sel(
        latitude=slice(LAT_MIN, LAT_MAX),
        longitude=slice(LON_MIN, LON_MAX)
    )
    return ds


def open_daily_peak_dataset() -> xr.Dataset:
    files = sorted(glob.glob(DAILY_PEAK_GLOB))
    if not files:
        raise FileNotFoundError(f"No files matched:\n{DAILY_PEAK_GLOB}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        preprocess=standardize_time_dim,
        engine="h5netcdf"
    )

    needed = ["wbt_daily_peak", TEMP_VAR, Q_VAR]
    missing = [v for v in needed if v not in ds.data_vars]
    if missing:
        raise ValueError(
            f"Missing required variables: {missing}\n"
            f"Available variables include: {list(ds.data_vars)}"
        )

    ds = ds.chunk({"time": -1})
    return ds


def load_phase_table():
    df = pd.read_csv(PHASE_CSV)

    if "time" not in df.columns:
        raise ValueError(
            f"Expected PHASE_CSV to have a 'time' column. "
            f"Available columns: {list(df.columns)}"
        )

    if "RONI" not in df.columns or "DMI" not in df.columns:
        raise ValueError(
            f"Expected PHASE_CSV to have 'RONI' and 'DMI'. "
            f"Available columns: {list(df.columns)}"
        )

    df["time"] = pd.to_datetime(df["time"])
    df["ym"] = df["time"].dt.to_period("M")

    df = (
        df.sort_values("time")
          .groupby("ym", as_index=False)
          .first()
          .copy()
    )

    df["year"] = df["ym"].dt.year.astype(int)
    df["month"] = df["ym"].dt.month.astype(int)

    df = df.rename(columns={
        "RONI": "RONI_raw",
        "DMI": "DMI_raw",
    })

    df = df.sort_values("ym").reset_index(drop=True)

    df["RONI_lagged"] = df["RONI_raw"].shift(ENSO_LAG)
    df["DMI_lagged"] = df["DMI_raw"].shift(IOD_LAG)

    df["enso_phase_lagged"] = df["RONI_lagged"].map(classify_enso_from_roni)
    df["iod_phase_lagged"] = df["DMI_lagged"].map(classify_iod_from_dmi)

    if USE_MONTHS is not None:
        df = df[df["month"].isin(USE_MONTHS)].copy()

    return df


def attach_monthly_phases(ds: xr.Dataset, phase_df: pd.DataFrame) -> xr.Dataset:
    time_index = pd.to_datetime(ds["time"].values)
    ym = pd.Series(time_index).dt.to_period("M")

    if USE_MONTHS is not None:
        keep = pd.Series(time_index).dt.month.isin(USE_MONTHS).values
        ds = ds.isel(time=np.where(keep)[0])
        time_index = pd.to_datetime(ds["time"].values)
        ym = pd.Series(time_index).dt.to_period("M")

    phase_lookup = phase_df.set_index("ym")

    enso_vals = ym.map(phase_lookup["enso_phase_lagged"]).to_numpy()
    iod_vals = ym.map(phase_lookup["iod_phase_lagged"]).to_numpy()

    ds = ds.assign_coords({
        "enso_phase_lagged": ("time", enso_vals),
        "iod_phase_lagged": ("time", iod_vals),
    })

    return ds


def phase_subset(ds: xr.Dataset, phase_coord: str, phase_label: str) -> xr.Dataset:
    mask = xr.DataArray(
        ds[phase_coord].values == phase_label,
        dims=("time",),
        coords={"time": ds["time"]}
    )
    out = ds.sel(time=mask)
    if out.sizes.get("time", 0) == 0:
        raise ValueError(f"No times found for {phase_coord} == {phase_label}")
    return out


def safe_quantile(da: xr.DataArray, q: float) -> xr.DataArray:
    if hasattr(da.data, "chunks") and da.chunks is not None:
        da = da.chunk({"time": -1})

    count = da.count("time")
    out = da.quantile(q, dim="time", skipna=True)
    out = out.where(count >= MIN_COUNT)

    if "quantile" in out.dims:
        out = out.squeeze("quantile", drop=True)

    return out


def compute_phase_p95_map(ds, phase_coord, phase_label, varname):
    sub = phase_subset(ds, phase_coord, phase_label)
    return safe_quantile(sub[varname], PCTL).rename(f"{varname}_{phase_label}_p95")


def nice_cbar_limit(*arrays, percentile=98):
    vals = []
    for arr in arrays:
        x = np.asarray(arr.values).ravel()
        x = x[np.isfinite(x)]
        if x.size:
            vals.append(x)

    if not vals:
        return 1.0

    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), percentile)
    return float(max(vmax, 0.25))

def nice_absolute_limits(*arrays, lower=2, upper=98):
    """
    Shared robust limits for absolute phase-specific fields.
    Uses all supplied arrays so each row has one common color scale.
    """
    vals = []

    for arr in arrays:
        x = np.asarray(arr.values).ravel()
        x = x[np.isfinite(x)]

        if x.size:
            vals.append(x)

    if not vals:
        return 0.0, 1.0

    vals = np.concatenate(vals)

    vmin = float(np.nanpercentile(vals, lower))
    vmax = float(np.nanpercentile(vals, upper))

    return vmin, vmax

def prep_for_pdf_mesh(field: xr.DataArray) -> xr.DataArray:
    """
    Sort coordinates so pcolormesh gets monotonic axes, which helps avoid
    PDF seam/striping artifacts.
    """
    out = field
    if "latitude" in out.coords:
        if np.any(np.diff(out["latitude"].values) < 0):
            out = out.sortby("latitude")
    if "longitude" in out.coords:
        out = out.sortby("longitude")
    return out

def add_map_features(ax, manuscript=False):
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6 if manuscript else 0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4 if manuscript else 0.5)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()

    if manuscript:
        gl.xlabel_style = {"size": 7}
        gl.ylabel_style = {"size": 7}


def add_panel_label(ax, label, manuscript=False):
    ax.text(
        0.02, 0.98, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9 if manuscript else 11,
        fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5),
        zorder=10
    )


def build_figure(
    enso_wbt_pos, enso_wbt_neg,
    iod_wbt_pos, iod_wbt_neg,
    enso_t_pos, enso_t_neg,
    iod_t_pos, iod_t_neg,
    enso_q_pos, enso_q_neg,
    iod_q_pos, iod_q_neg,
    manuscript=False
):
    proj = ccrs.PlateCarree()

    figsize = (7.1, 6.2) if manuscript else (15, 10)

    fig, axes = plt.subplots(
        3, 4,
        figsize=figsize,
        subplot_kw={"projection": proj},
        constrained_layout=True
    )

    # ---------------------------------------------------------
    # q: kg/kg -> g/kg
    # ---------------------------------------------------------
    enso_q_pos_gkg = (enso_q_pos * 1000.0).rename("enso_q_pos_gkg")
    enso_q_neg_gkg = (enso_q_neg * 1000.0).rename("enso_q_neg_gkg")
    iod_q_pos_gkg  = (iod_q_pos  * 1000.0).rename("iod_q_pos_gkg")
    iod_q_neg_gkg  = (iod_q_neg  * 1000.0).rename("iod_q_neg_gkg")

    # ---------------------------------------------------------
    # Shared color limits WITHIN each row
    # ---------------------------------------------------------
    wbt_vmin, wbt_vmax = nice_absolute_limits(
        enso_wbt_pos,
        enso_wbt_neg,
        iod_wbt_pos,
        iod_wbt_neg
    )

    t_vmin, t_vmax = nice_absolute_limits(
        enso_t_pos,
        enso_t_neg,
        iod_t_pos,
        iod_t_neg
    )

    q_vmin, q_vmax = nice_absolute_limits(
        enso_q_pos_gkg,
        enso_q_neg_gkg,
        iod_q_pos_gkg,
        iod_q_neg_gkg
    )

    wbt_norm = mpl.colors.Normalize(vmin=wbt_vmin, vmax=wbt_vmax)
    t_norm   = mpl.colors.Normalize(vmin=t_vmin, vmax=t_vmax)
    q_norm   = mpl.colors.Normalize(vmin=q_vmin, vmax=q_vmax)

    # ---------------------------------------------------------
    # Fields
    #
    # Column order:
    #   El Niño | La Niña | pIOD | nIOD
    # ---------------------------------------------------------
    wbt_fields = [
        enso_wbt_pos,   # El Niño
        enso_wbt_neg,   # La Niña
        iod_wbt_pos,    # pIOD
        iod_wbt_neg,    # nIOD
    ]

    t_fields = [
        enso_t_pos,
        enso_t_neg,
        iod_t_pos,
        iod_t_neg,
    ]

    q_fields = [
        enso_q_pos_gkg,
        enso_q_neg_gkg,
        iod_q_pos_gkg,
        iod_q_neg_gkg,
    ]

    # ---------------------------------------------------------
    # Plot WBT row
    # ---------------------------------------------------------
    mappable_wbt = None

    for j, field in enumerate(wbt_fields):

        ax = axes[0, j]
        add_map_features(ax, manuscript=manuscript)

        plot_field = prep_for_pdf_mesh(field) if manuscript else field

        mappable_wbt = ax.pcolormesh(
            plot_field["longitude"],
            plot_field["latitude"],
            plot_field,
            transform=proj,
            cmap="viridis",
            norm=wbt_norm,
            shading="nearest" if manuscript else "auto",
            antialiased=False if manuscript else True,
            linewidth=0.0,
            edgecolors="none",
            rasterized=manuscript,
        )

        add_panel_label(
            ax,
            PANEL_LABELS[j],
            manuscript=manuscript
        )

    # ---------------------------------------------------------
    # Plot Ta row
    # ---------------------------------------------------------
    mappable_t = None

    for j, field in enumerate(t_fields):

        ax = axes[1, j]
        add_map_features(ax, manuscript=manuscript)

        plot_field = prep_for_pdf_mesh(field) if manuscript else field

        mappable_t = ax.pcolormesh(
            plot_field["longitude"],
            plot_field["latitude"],
            plot_field,
            transform=proj,
            cmap="viridis",
            norm=t_norm,
            shading="nearest" if manuscript else "auto",
            antialiased=False if manuscript else True,
            linewidth=0.0,
            edgecolors="none",
            rasterized=manuscript,
        )

        add_panel_label(
            ax,
            PANEL_LABELS[4 + j],
            manuscript=manuscript
        )

    # ---------------------------------------------------------
    # Plot q row
    # ---------------------------------------------------------
    mappable_q = None

    for j, field in enumerate(q_fields):

        ax = axes[2, j]
        add_map_features(ax, manuscript=manuscript)

        plot_field = prep_for_pdf_mesh(field) if manuscript else field

        mappable_q = ax.pcolormesh(
            plot_field["longitude"],
            plot_field["latitude"],
            plot_field,
            transform=proj,
            cmap="YlGnBu",
            norm=q_norm,
            shading="nearest" if manuscript else "auto",
            antialiased=False if manuscript else True,
            linewidth=0.0,
            edgecolors="none",
            rasterized=manuscript,
        )

        add_panel_label(
            ax,
            PANEL_LABELS[8 + j],
            manuscript=manuscript
        )

    # ---------------------------------------------------------
    # Column headers
    # ---------------------------------------------------------
    col_fs = 9 if manuscript else 13

    column_titles = [
        "El Niño",
        "La Niña",
        "pIOD",
        "nIOD",
    ]

    for j, title in enumerate(column_titles):
        axes[0, j].set_title(
            title,
            fontsize=col_fs,
            pad=8,
            fontweight="bold"
        )

    # ---------------------------------------------------------
    # Row labels
    # ---------------------------------------------------------
    row_fs = 9 if manuscript else 12

    row_labels = [
        r"$T_w$",
        r"$T_a$ at max $T_w$",
        r"$q$ at max $T_w$",
    ]

    for i, label in enumerate(row_labels):

        axes[i, 0].text(
            -0.23 if manuscript else -0.16,
            0.5,
            label,
            transform=axes[i, 0].transAxes,
            rotation=90,
            va="center",
            ha="center",
            fontsize=row_fs,
            fontweight="bold"
        )

    # ---------------------------------------------------------
    # One colorbar per row
    # ---------------------------------------------------------
    cbar_wbt = fig.colorbar(
        mappable_wbt,
        ax=list(axes[0, :]),
        shrink=0.94,
        pad=0.02
    )
    cbar_wbt.set_label(r"$p95$ $T_w$ (°C)")

    cbar_t = fig.colorbar(
        mappable_t,
        ax=list(axes[1, :]),
        shrink=0.94,
        pad=0.02
    )
    cbar_t.set_label(r"$p95$ $T_a$ at max $T_w$ (°C)")

    cbar_q = fig.colorbar(
        mappable_q,
        ax=list(axes[2, :]),
        shrink=0.94,
        pad=0.02
    )
    cbar_q.set_label(r"$p95$ $q$ at max $T_w$ (g/kg)")

    if manuscript:

        cbar_wbt.ax.tick_params(labelsize=7)
        cbar_t.ax.tick_params(labelsize=7)
        cbar_q.ax.tick_params(labelsize=7)

    else:

        fig.suptitle(
            f"Phase-specific p95 daily peak-state fields\n"
            f"ENSO lag = {ENSO_LAG} months, "
            f"IOD lag = {IOD_LAG} month",
            fontsize=14
        )

    return fig


def save_main_figure(
    enso_wbt_pos, enso_wbt_neg,
    iod_wbt_pos, iod_wbt_neg,
    enso_t_pos, enso_t_neg,
    iod_t_pos, iod_t_neg,
    enso_q_pos, enso_q_neg,
    iod_q_pos, iod_q_neg
):
    # PNG
    fig = build_figure(
        enso_wbt_pos, enso_wbt_neg,
        iod_wbt_pos, iod_wbt_neg,
        enso_t_pos, enso_t_neg,
        iod_t_pos, iod_t_neg,
        enso_q_pos, enso_q_neg,
        iod_q_pos, iod_q_neg,
        manuscript=False
    )

    fig.savefig(
        OUT_DIR / "phase_p95_wbt_t_q_12panel_lagged.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    # PDF manuscript version
    with mpl.rc_context(ERL_RC):

        fig = build_figure(
            enso_wbt_pos, enso_wbt_neg,
            iod_wbt_pos, iod_wbt_neg,
            enso_t_pos, enso_t_neg,
            iod_t_pos, iod_t_neg,
            enso_q_pos, enso_q_neg,
            iod_q_pos, iod_q_neg,
            manuscript=True
        )

        fig.savefig(
            OUT_DIR / "phase_p95_wbt_t_q_12panel_lagged_manuscript.pdf",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.02
        )

        plt.close(fig)


# ============================================================
# MAIN
# ============================================================
def main():
    print("Opening DailyPeakState files...")
    ds = open_daily_peak_dataset()

    print("Loading and lagging monthly RONI/DMI table...")
    phase_df = load_phase_table()

    print("Attaching monthly lagged phases to daily data...")
    ds = attach_monthly_phases(ds, phase_df)
    
    print("Defining phase-specific p95 maps...")
    
    # ENSO
    enso_wbt_neg = compute_phase_p95_map(
        ds, "enso_phase_lagged", "La Nina", "wbt_daily_peak"
    )
    enso_wbt_pos = compute_phase_p95_map(
        ds, "enso_phase_lagged", "El Nino", "wbt_daily_peak"
    )
    
    enso_t_neg = compute_phase_p95_map(
        ds, "enso_phase_lagged", "La Nina", TEMP_VAR
    )
    enso_t_pos = compute_phase_p95_map(
        ds, "enso_phase_lagged", "El Nino", TEMP_VAR
    )
    
    enso_q_neg = compute_phase_p95_map(
        ds, "enso_phase_lagged", "La Nina", Q_VAR
    )
    enso_q_pos = compute_phase_p95_map(
        ds, "enso_phase_lagged", "El Nino", Q_VAR
    )
    
    # IOD
    iod_wbt_neg = compute_phase_p95_map(
        ds, "iod_phase_lagged", "nIOD", "wbt_daily_peak"
    )
    iod_wbt_pos = compute_phase_p95_map(
        ds, "iod_phase_lagged", "pIOD", "wbt_daily_peak"
    )
    
    iod_t_neg = compute_phase_p95_map(
        ds, "iod_phase_lagged", "nIOD", TEMP_VAR
    )
    iod_t_pos = compute_phase_p95_map(
        ds, "iod_phase_lagged", "pIOD", TEMP_VAR
    )
    
    iod_q_neg = compute_phase_p95_map(
        ds, "iod_phase_lagged", "nIOD", Q_VAR
    )
    iod_q_pos = compute_phase_p95_map(
        ds, "iod_phase_lagged", "pIOD", Q_VAR
    )
    
    print("Computing final phase maps...")
    
    maps = xr.Dataset({
        "enso_wbt_pos": enso_wbt_pos,
        "enso_wbt_neg": enso_wbt_neg,
        "enso_t_pos": enso_t_pos,
        "enso_t_neg": enso_t_neg,
        "enso_q_pos": enso_q_pos,
        "enso_q_neg": enso_q_neg,
    
        "iod_wbt_pos": iod_wbt_pos,
        "iod_wbt_neg": iod_wbt_neg,
        "iod_t_pos": iod_t_pos,
        "iod_t_neg": iod_t_neg,
        "iod_q_pos": iod_q_pos,
        "iod_q_neg": iod_q_neg,
    }).compute()
    
    enso_wbt_pos = maps["enso_wbt_pos"]
    enso_wbt_neg = maps["enso_wbt_neg"]
    enso_t_pos   = maps["enso_t_pos"]
    enso_t_neg   = maps["enso_t_neg"]
    enso_q_pos   = maps["enso_q_pos"]
    enso_q_neg   = maps["enso_q_neg"]
    
    iod_wbt_pos = maps["iod_wbt_pos"]
    iod_wbt_neg = maps["iod_wbt_neg"]
    iod_t_pos   = maps["iod_t_pos"]
    iod_t_neg   = maps["iod_t_neg"]
    iod_q_pos   = maps["iod_q_pos"]
    iod_q_neg   = maps["iod_q_neg"]

    print("Saving phase-specific PNG + manuscript PDF...")

    save_main_figure(
        enso_wbt_pos, enso_wbt_neg,
        iod_wbt_pos, iod_wbt_neg,
        enso_t_pos, enso_t_neg,
        iod_t_pos, iod_t_neg,
        enso_q_pos, enso_q_neg,
        iod_q_pos, iod_q_neg
    )

    debug_ds = xr.Dataset({

        # ENSO
        "el_nino_wbt_p95": enso_wbt_pos,
        "la_nina_wbt_p95": enso_wbt_neg,

        "el_nino_t_p95": enso_t_pos,
        "la_nina_t_p95": enso_t_neg,

        "el_nino_q_p95": enso_q_pos,
        "la_nina_q_p95": enso_q_neg,

        # IOD
        "pIOD_wbt_p95": iod_wbt_pos,
        "nIOD_wbt_p95": iod_wbt_neg,

        "pIOD_t_p95": iod_t_pos,
        "nIOD_t_p95": iod_t_neg,

        "pIOD_q_p95": iod_q_pos,
        "nIOD_q_p95": iod_q_neg,
    })

    debug_ds.to_netcdf(
        OUT_DIR / "phase_map_products_individual_lagged.nc"
    )

    print("\nDone.")
    print(f"Saved to: {OUT_DIR}")


if __name__ == "__main__":
    main()

Opening DailyPeakState files...
Loading and lagging monthly RONI/DMI table...
Attaching monthly lagged phases to daily data...
Defining phase-specific p95 maps...
Computing final phase maps...
Saving phase-specific PNG + manuscript PDF...

Done.
Saved to: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/figures/phase_maps
